In [1]:
import requests
from pathlib import Path

url = "https://opendata.5t.torino.it/get_fdt"

output = Path(r"C:\Users\reyha\Desktop\turin-mobility-ai\data\raw\traffic.xml")
output.parent.mkdir(parents=True, exist_ok=True)

response = requests.get(url, timeout=30)

print("Status:", response.status_code)

if response.status_code == 200:
    output.write_bytes(response.content)
    print("Saved to:", output)
else:
    print("Download failed:", response.status_code)

Status: 200
Saved to: C:\Users\reyha\Desktop\turin-mobility-ai\data\raw\traffic.xml


Traffic

In [2]:


url = "https://opendata.5t.torino.it/get_pk"

output = Path(r"C:\Users\reyha\Desktop\turin-mobility-ai\data\raw\parking.xml")
output.parent.mkdir(parents=True, exist_ok=True)

response = requests.get(url, timeout=30)

print("Status:", response.status_code)

if response.status_code == 200:
    output.write_bytes(response.content)
    print("Saved to:", output)
else:
    print("Download failed:", response.status_code)

Status: 200
Saved to: C:\Users\reyha\Desktop\turin-mobility-ai\data\raw\parking.xml


In [3]:
import xml.etree.ElementTree as ET


traffic_path = Path(r"C:\Users\reyha\Desktop\turin-mobility-ai\data\raw\traffic.xml")

tree = ET.parse(traffic_path)
root = tree.getroot()

print("Root tag:", root.tag)
print("Number of first-level elements:", len(root))

Root tag: {https://simone.5t.torino.it/ns/traffic_data.xsd}traffic_data
Number of first-level elements: 113


In [4]:
for child in list(root)[:5]:
    print("TAG:", child.tag)
    print("ATTRIBUTES:", child.attrib)
    print("TEXT:", child.text)
    print("---")

TAG: {https://simone.5t.torino.it/ns/traffic_data.xsd}location_reference
ATTRIBUTES: {}
TEXT: None
---
TAG: {https://simone.5t.torino.it/ns/traffic_data.xsd}FDT_data
ATTRIBUTES: {'lcd1': '40121', 'Road_LCD': '40118', 'Road_name': 'Corso Regina Margherita(TO)', 'offset': '239', 'direction': 'positive', 'lat': '45.096231', 'lng': '7.625643', 'accuracy': '0', 'period': '5'}
TEXT: None
---
TAG: {https://simone.5t.torino.it/ns/traffic_data.xsd}FDT_data
ATTRIBUTES: {'lcd1': '40123', 'Road_LCD': '40118', 'Road_name': 'Corso Regina Margherita(TO)', 'offset': '1555', 'direction': 'negative', 'lat': '45.093256', 'lng': '7.63375', 'accuracy': '0', 'period': '5'}
TEXT: None
---
TAG: {https://simone.5t.torino.it/ns/traffic_data.xsd}FDT_data
ATTRIBUTES: {'lcd1': '39983', 'Road_LCD': '39980', 'Road_name': 'Corso Allamano(TO)', 'offset': '563', 'direction': 'positive', 'lat': '45.0507', 'lng': '7.6225', 'accuracy': '95', 'period': '5'}
TEXT: None
---
TAG: {https://simone.5t.torino.it/ns/traffic_data.x

In [5]:
first_record = list(root)[0]

print("Record tag:", first_record.tag)
print("Record attributes:", first_record.attrib)

for element in first_record:
    print(
        "Child:",
        element.tag,
        "Attributes:",
        element.attrib
    )

Record tag: {https://simone.5t.torino.it/ns/traffic_data.xsd}location_reference
Record attributes: {}
Child: {https://simone.5t.torino.it/ns/traffic_data.xsd}WGS84_info Attributes: {}


In [6]:
record = list(root)[1]

print("Road information:")
print(record.attrib)

print("Traffic information:")
for item in record:
    print(item.attrib)

Road information:
{'lcd1': '40121', 'Road_LCD': '40118', 'Road_name': 'Corso Regina Margherita(TO)', 'offset': '239', 'direction': 'positive', 'lat': '45.096231', 'lng': '7.625643', 'accuracy': '0', 'period': '5'}
Traffic information:
{'flow': '0', 'speed': '0'}


In [7]:
record = list(root)[1]
print("Road information:")
print(record.attrib)

Road information:
{'lcd1': '40121', 'Road_LCD': '40118', 'Road_name': 'Corso Regina Margherita(TO)', 'offset': '239', 'direction': 'positive', 'lat': '45.096231', 'lng': '7.625643', 'accuracy': '0', 'period': '5'}


In [8]:
for item in record:
    print(item.tag)
    print(item.attrib)

{https://simone.5t.torino.it/ns/traffic_data.xsd}speedflow
{'flow': '0', 'speed': '0'}


In [9]:
import pandas as pd

rows = []

for record in list(root)[1:]:
    
    road = record.attrib
    traffic = list(record)[0].attrib

    rows.append({
        "sensor_id": road.get("lcd1"),
        "road_name": road.get("Road_name"),
        "direction": road.get("direction"),
        "lat": road.get("lat"),
        "lng": road.get("lng"),
        "period": road.get("period"),
        "flow": traffic.get("flow"),
        "speed": traffic.get("speed"),
        "road_id": road.get("Road_LCD"),
         "offset": road.get("offset")
    })

df = pd.DataFrame(rows)

df.head()

,sensor_id,road_name,direction,lat,lng,period,flow,speed,road_id,offset
0,40121,Corso Regina Margherita(TO),positive,45.096231,7.625643,5,0,0,40118,239
1,40123,Corso Regina Margherita(TO),negative,45.093256,7.63375,5,0,0,40118,1555
2,39983,Corso Allamano(TO),positive,45.0507,7.6225,5,156,50.29,39980,563
3,40071,Corso Moncalieri(TO),positive,45.035647,7.680685,5,0,0,40070,1836
4,40253,Strada Di Settimo(TO),positive,45.103604,7.725984,5,264,59.51,40251,511


In [10]:
df.shape

(112, 10)

In [11]:
df.columns

Index(['sensor_id', 'road_name', 'direction', 'lat', 'lng', 'period', 'flow',
       'speed', 'road_id', 'offset'],
      dtype='object')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sensor_id  112 non-null    object
 1   road_name  112 non-null    object
 2   direction  112 non-null    object
 3   lat        112 non-null    object
 4   lng        112 non-null    object
 5   period     112 non-null    object
 6   flow       112 non-null    object
 7   speed      112 non-null    object
 8   road_id    112 non-null    object
 9   offset     112 non-null    object
dtypes: object(10)
memory usage: 8.9+ KB


In [13]:
df["sensor_id"] = df["sensor_id"].astype(str)
df["road_id"] = df["road_id"].astype(str)

In [14]:
df["offset"] = pd.to_numeric(df["offset"])

In [15]:
df_float=["lat","lng","speed"]
for col in df_float:
    df[col]=df[col].astype(float)

In [16]:
df_int=["period","flow"]
for col in df_int:
    df[col]=df[col].astype(int)

In [17]:
df["flow"].head()

0      0
1      0
2    156
3      0
4    264
Name: flow, dtype: int64

In [18]:
df["flow"].isnull().sum()

np.int64(0)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sensor_id  112 non-null    object 
 1   road_name  112 non-null    object 
 2   direction  112 non-null    object 
 3   lat        112 non-null    float64
 4   lng        112 non-null    float64
 5   period     112 non-null    int64  
 6   flow       112 non-null    int64  
 7   speed      112 non-null    float64
 8   road_id    112 non-null    object 
 9   offset     112 non-null    int64  
dtypes: float64(3), int64(3), object(4)
memory usage: 8.9+ KB


In [20]:
df.isnull().sum()

sensor_id    0
road_name    0
direction    0
lat          0
lng          0
period       0
flow         0
speed        0
road_id      0
offset       0
dtype: int64

In [21]:
df.duplicated().sum()

np.int64(0)

In [22]:
df.head()

,sensor_id,road_name,direction,lat,lng,period,flow,speed,road_id,offset
0,40121,Corso Regina Margherita(TO),positive,45.096231,7.625643,5,0,0.00,40118,239
1,40123,Corso Regina Margherita(TO),negative,45.093256,7.633750,5,0,0.00,40118,1555
2,39983,Corso Allamano(TO),positive,45.050700,7.622500,5,156,50.29,39980,563
3,40071,Corso Moncalieri(TO),positive,45.035647,7.680685,5,0,0.00,40070,1836
4,40253,Strada Di Settimo(TO),positive,45.103604,7.725984,5,264,59.51,40251,511


In [23]:
df["period"].head()

0    5
1    5
2    5
3    5
4    5
Name: period, dtype: int64

In [24]:
df.duplicated(subset=["sensor_id", "direction"]).sum()

np.int64(1)

In [25]:
df.duplicated(
    subset=["sensor_id", "direction", "offset"]
).sum()

np.int64(0)

In [26]:
duplicates = df[df.duplicated(
    subset=["sensor_id", "direction"],
    keep=False
)]

duplicates

,sensor_id,road_name,direction,lat,lng,period,flow,speed,road_id,offset
14,40189,Corso Unita' D'italia(TO),positive,45.01876,7.669320,5,1200,56.76,40188,160
23,40189,Corso Unita' D'italia(TO),positive,45.02586,7.671667,5,1488,58.00,40188,970


In [53]:
df.describe()

,lat,lng,period,flow,speed,offset,collected_at
count,112.000000,112.000000,112.0,112.000000,112.000000,112.00000,112
mean,45.097682,8.055115,5.0,252.428571,58.939643,2428.00000,2026-09-15 22:25:33.688585
min,44.394930,7.484170,5.0,0.000000,0.000000,96.00000,2026-09-15 22:25:33.688586
25%,44.917210,7.669308,5.0,60.000000,50.822500,740.00000,2026-09-15 22:25:33.688586
50%,45.070407,7.974130,5.0,120.000000,65.000000,1750.00000,2026-09-15 22:25:33.688586
75%,45.338738,8.434643,5.0,255.000000,76.000000,3512.50000,2026-09-15 22:25:33.688586
max,45.931380,8.895240,5.0,4656.000000,102.000000,10400.00000,2026-09-15 22:25:33.688586
std,0.343321,0.423215,0.0,507.695406,26.264790,2180.79302,NaN


In [27]:
import xml.etree.ElementTree as ET

tree_parking = ET.parse(r"C:\Users\reyha\Desktop\turin-mobility-ai\data\raw\parking.xml")
root_parking = tree_parking.getroot()

print(root_parking.tag)
print(len(root_parking))

{https://simone.5t.torino.it/ns/traffic_data.xsd}traffic_data
42


In [28]:
for item in list(root_parking)[:5]:
    print(item.tag)
    print(item.attrib)

{https://simone.5t.torino.it/ns/traffic_data.xsd}location_reference
{}
{https://simone.5t.torino.it/ns/traffic_data.xsd}PK_data
{'Name': 'BODONI', 'ID': '2', 'status': '1', 'Total': '460', 'Free': '258', 'tendence': '1', 'lat': '45.063553', 'lng': '7.683574'}
{https://simone.5t.torino.it/ns/traffic_data.xsd}PK_data
{'Name': 'BOLZANO', 'ID': '3', 'status': '1', 'Total': '858', 'Free': '398', 'tendence': '1', 'lat': '45.072478', 'lng': '7.667162'}
{https://simone.5t.torino.it/ns/traffic_data.xsd}PK_data
{'Name': "D'AZEGLIO GALILEI", 'ID': '4', 'status': '1', 'Total': '223', 'Free': '149', 'tendence': '-1', 'lat': '45.042894', 'lng': '7.677542'}
{https://simone.5t.torino.it/ns/traffic_data.xsd}PK_data
{'Name': 'EMANUELE FILIBERTO', 'ID': '5', 'status': '1', 'Total': '110', 'Free': '28', 'tendence': '1', 'lat': '45.076661393', 'lng': '7.68030909027'}


In [29]:
parking_record = list(root_parking)[0]

print(parking_record.attrib)

{}


In [30]:
for item in parking_record:
    print(item.tag)
    print(item.attrib)
    print(item.text)

{https://simone.5t.torino.it/ns/traffic_data.xsd}tmc_info
{'tabcd': '1', 'cid': '25'}
None


In [31]:
parking_record = list(root_parking)[1]

print(parking_record.attrib)

{'Name': 'BODONI', 'ID': '2', 'status': '1', 'Total': '460', 'Free': '258', 'tendence': '1', 'lat': '45.063553', 'lng': '7.683574'}


In [32]:
import pandas as pd

rows = []

for record in list(root_parking)[1:]:
    data = record.attrib

    rows.append({
        "name": data.get("Name"),
        "id": data.get("ID"),
        "status": data.get("status"),
        "total": data.get("Total"),
        "free": data.get("Free"),
        "tendence": data.get("tendence"),
        "lat": data.get("lat"),
        "lng": data.get("lng")
    })

parking_df = pd.DataFrame(rows)

parking_df.head()

,name,id,status,total,free,tendence,lat,lng
0,BODONI,2,1,460,258,1,45.063553,7.683574
1,BOLZANO,3,1,858,398,1,45.072478,7.667162
2,D'AZEGLIO GALILEI,4,1,223,149,-1,45.042894,7.677542
3,EMANUELE FILIBERTO,5,1,110,28,1,45.076661393,7.68030909027
4,GALILEO FERRARIS,6,1,276,213,1,45.067451,7.672192


In [33]:
parking_df.shape

(41, 8)

In [34]:
parking_df.columns

Index(['name', 'id', 'status', 'total', 'free', 'tendence', 'lat', 'lng'], dtype='object')

In [35]:
parking_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      41 non-null     object
 1   id        41 non-null     object
 2   status    41 non-null     object
 3   total     41 non-null     object
 4   free      37 non-null     object
 5   tendence  41 non-null     object
 6   lat       41 non-null     object
 7   lng       41 non-null     object
dtypes: object(8)
memory usage: 2.7+ KB


In [36]:
parking_df["free"] = parking_df["free"].astype("Int64")

In [37]:
parking_columns=["id","status","total","tendence","lat","lng"]
for col in parking_columns:
    parking_df[col]=pd.to_numeric(parking_df[col])

In [38]:
parking_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      41 non-null     object 
 1   id        41 non-null     int64  
 2   status    41 non-null     int64  
 3   total     41 non-null     int64  
 4   free      37 non-null     Int64  
 5   tendence  41 non-null     int64  
 6   lat       41 non-null     float64
 7   lng       41 non-null     float64
dtypes: Int64(1), float64(2), int64(4), object(1)
memory usage: 2.7+ KB


In [39]:
parking_df.isnull().sum()

name        0
id          0
status      0
total       0
free        4
tendence    0
lat         0
lng         0
dtype: int64

In [40]:
parking_df[parking_df["free"].isnull()]

,name,id,status,total,free,tendence,lat,lng
15,RACCONIGI,17,0,100,<NA>,1,45.069825,7.646689
19,FERMI,24,0,300,<NA>,1,45.076464,7.591056
24,V PADIGLIONE,31,0,327,<NA>,1,45.050827,7.682977
26,UNIONE SOVIETICA,40,0,150,<NA>,1,45.011920,7.622910


In [41]:
parking_df.duplicated().sum()

np.int64(0)

In [ ]:
parking_df.describe()

,id,status,total,free,tendence,lat,lng,collected_at
count,41.000000,41.000000,41.000000,37.0,41.000000,41.000000,41.000000,41
mean,29.341463,0.902439,384.365854,287.243243,0.560976,45.061068,7.669610,2026-09-15 22:25:33.688586
min,2.000000,0.000000,57.000000,28.0,-1.000000,45.011920,7.591056,2026-09-15 22:25:33.688586
25%,12.000000,1.000000,150.000000,108.0,1.000000,45.042894,7.661091,2026-09-15 22:25:33.688586
50%,25.000000,1.000000,300.000000,183.0,1.000000,45.065041,7.673480,2026-09-15 22:25:33.688586
75%,50.000000,1.000000,457.000000,309.0,1.000000,45.073390,7.683419,2026-09-15 22:25:33.688586
max,66.000000,1.000000,3100.000000,2959.0,1.000000,45.120578,7.716923,2026-09-15 22:25:33.688586
std,19.940674,0.300406,475.933649,474.246268,0.838116,0.020817,0.024351,NaN


: 

timestamp

In [42]:
collected_at = pd.Timestamp.now()

df["collected_at"] = collected_at
parking_df["collected_at"] = collected_at

In [43]:
df.head()

,sensor_id,road_name,direction,lat,lng,period,flow,speed,road_id,offset,collected_at
0,40121,Corso Regina Margherita(TO),positive,45.096231,7.625643,5,0,0.00,40118,239,2026-09-15 22:25:33.688586
1,40123,Corso Regina Margherita(TO),negative,45.093256,7.633750,5,0,0.00,40118,1555,2026-09-15 22:25:33.688586
2,39983,Corso Allamano(TO),positive,45.050700,7.622500,5,156,50.29,39980,563,2026-09-15 22:25:33.688586
3,40071,Corso Moncalieri(TO),positive,45.035647,7.680685,5,0,0.00,40070,1836,2026-09-15 22:25:33.688586
4,40253,Strada Di Settimo(TO),positive,45.103604,7.725984,5,264,59.51,40251,511,2026-09-15 22:25:33.688586


In [44]:
parking_df.head()

,name,id,status,total,free,tendence,lat,lng,collected_at
0,BODONI,2,1,460,258,1,45.063553,7.683574,2026-09-15 22:25:33.688586
1,BOLZANO,3,1,858,398,1,45.072478,7.667162,2026-09-15 22:25:33.688586
2,D'AZEGLIO GALILEI,4,1,223,149,-1,45.042894,7.677542,2026-09-15 22:25:33.688586
3,EMANUELE FILIBERTO,5,1,110,28,1,45.076661,7.680309,2026-09-15 22:25:33.688586
4,GALILEO FERRARIS,6,1,276,213,1,45.067451,7.672192,2026-09-15 22:25:33.688586


In [45]:
df.to_csv(r"C:\Users\reyha\Desktop\turin-mobility-ai\processed\traffic_clean.csv", index=False)

parking_df.to_csv(r"C:\Users\reyha\Desktop\turin-mobility-ai\processed\parking_clean.csv", index=False)

database structure we have two tabel for parking and traffic

In [46]:
parking_observations = parking_df[
    ["id", "status", "free", "tendence", "collected_at"]
].copy()

parking_observations = parking_observations.rename(
    columns={"id": "parking_id"}
)

In [47]:
parking_observations["free"] = parking_observations["free"].astype("Int64")

In [48]:
parking_observations.head()

,parking_id,status,free,tendence,collected_at
0,2,1,258,1,2026-09-15 22:25:33.688586
1,3,1,398,1,2026-09-15 22:25:33.688586
2,4,1,149,-1,2026-09-15 22:25:33.688586
3,5,1,28,1,2026-09-15 22:25:33.688586
4,6,1,213,1,2026-09-15 22:25:33.688586


In [49]:


parking_observations.to_csv(r"C:\Users\reyha\Desktop\turin-mobility-ai\processed\parking_observations.csv",index=False)

In [50]:
traffic_observations = df[
    ["sensor_id", "direction", "offset", "period", "flow", "speed", "collected_at"]
].copy()

In [51]:
traffic_observations.to_csv(
    r"C:\Users\reyha\Desktop\turin-mobility-ai\processed\traffic_observations.csv",
    index=False
)

In [52]:
print("Traffic shape:", df.shape)
print("Parking shape:", parking_df.shape)

Traffic shape: (112, 11)
Parking shape: (41, 9)
